In [ ]:
import time         # to measure ray tracing time
import torch
import numpy as np
import matplotlib.pyplot as plt
from plotting_tools import plot_intensity_images
from VolumeRaytraceLFM.abstract_classes import BackEnds
from VolumeRaytraceLFM.birefringence_implementations import  (
    BirefringentVolume,
    BirefringentRaytraceLFM,
    JonesMatrixGenerators
)

In [ ]:
def retardance(JM):
    '''Phase delay introduced between the fast and slow axis in a Jones Matrix'''
    # if backend == BackEnds.NUMPY:
    e1, e2 = np.linalg.eigvals(JM)
    phase_diff = np.angle(e1) - np.angle(e2)
    retardance = np.abs(phase_diff)
    # elif backend == BackEnds.PYTORCH:
    #     x = torch.linalg.eigvals(JM)
    #     retardance = (torch.angle(x[:,1]) - torch.angle(x[:,0])).abs()
    # else:
    #     raise NotImplementedError
    return retardance

def azimuth(JM):
    '''Rotation angle of the fast axis (neg phase)'''
    diag_sum = JM[0, 0] + JM[1, 1]
    diag_diff = JM[1, 1] - JM[0, 0]
    off_diag_sum = JM[0, 1] + JM[1, 0]
    a = np.imag(diag_diff / diag_sum)
    b = np.imag(off_diag_sum / diag_sum)
    # if np.isclose(np.abs(a), 0.0):
    #     a = 0.0
    # if np.isclose(np.abs(b), 0.0):
    #     b = 0.0
    azimuth = np.arctan2(-b, -a) / 2 + np.pi / 2
    # if np.isclose(azimuth,np.pi):
    #     azimuth = 0.0
    # elif self.backend == BackEnds.PYTORCH:
    #     diag_sum = (JM[:, 0, 0] + JM[:, 1, 1])
    #     diag_diff = (JM[:, 1, 1] - JM[: ,0, 0])
    #     off_diag_sum = JM[:, 0, 1] + JM[:, 1, 0]
    #     a = (diag_diff / diag_sum).imag
    #     b = (off_diag_sum / diag_sum).imag
    #     # atan2 with zero entries causes nan in backward, so let's filter them out
    #     azimuth = torch.zeros_like(a)
    #     zero_a_b = torch.isclose(a,torch.zeros([1],dtype=a.dtype, device=a.device)).bitwise_and(torch.isclose(b,torch.zeros([1],dtype=b.dtype, device=a.device)))
    #     azimuth[~zero_a_b] = torch.arctan2(-b[~zero_a_b], -a[~zero_a_b]) / 2.0 + torch.pi / 2.0
    #     # TODO: if output azimuth is pi, make it 0 and vice-versa (arctan2 bug)
    #     # zero_index = torch.isclose(azimuth, torch.zeros([1]), atol=1e-5)
    #     # pi_index = torch.isclose(azimuth, torch.tensor(torch.pi), atol=1e-5)
    #     # azimuth[zero_index] = torch.pi
    #     # azimuth[pi_index] = 0
    return azimuth

In [ ]:
def voxRayJM(Delta_n, opticAxis, rayDir, ell, wavelength):
    '''Compute Jones matrix associated with a particular ray and voxel combination'''
    # if self.backend == BackEnds.NUMPY:
    # Azimuth is the angle of the slow axis of retardance.
    azim = np.arctan2(np.dot(opticAxis, rayDir[1]), np.dot(opticAxis, rayDir[2]))
    if Delta_n == 0:
        azim = 0
    elif Delta_n < 0:
        azim = azim + np.pi / 2
    # print(f"Azimuth angle of index ellipsoid is
    #   {np.around(np.rad2deg(azim), decimals=0)} degrees.")
    ret = abs(Delta_n) * (1 - np.dot(opticAxis, rayDir[0]) ** 2) * 2 * np.pi * ell / wavelength
    # print(f"Accumulated retardance from index ellipsoid is
    #   {np.around(np.rad2deg(ret), decimals=0)} ~ {int(np.rad2deg(ret)) % 360} degrees.")

    # TODO: compare speed
    # The following series of operations is an equivalent method as
    # offdiag = 1j * np.sin(2 * azim) * np.sin(ret / 2)
    # diag1 = np.cos(ret / 2) + 1j * np.cos(2 * azim) * np.sin(ret / 2)
    # diag2 = np.conj(diag1)
    # JM = np.array([[diag1, offdiag], [offdiag, diag2]])
    JM = JonesMatrixGenerators.linear_retarder(ret, azim)
#   elif self.backend == BackEnds.PYTORCH:
#     n_voxels = opticAxis.shape[0]
#     if not torch.is_tensor(opticAxis):
#         opticAxis = torch.from_numpy(opticAxis).to(Delta_n.device)

#     # Dot product of optical axis and 3 ray-direction vectors
#     OA_dot_rayDir = torch.linalg.vecdot(opticAxis, rayDir)

#     # Azimuth is the angle of the slow axis of retardance.
#     azim = 2 * torch.arctan2(OA_dot_rayDir[1,:], OA_dot_rayDir[2,:])
#     ret = abs(Delta_n) * (1 - OA_dot_rayDir[0,:] ** 2) * torch.pi * ell / wavelength

#     # The following series of operations is an equivalent, but more efficient method as
#     #   JM = JonesMatrixGenerators.linear_retarder(ret, azim, self.backend)
#     offdiag = 1j * torch.sin(azim) * torch.sin(ret)
#     diag1 = torch.cos(ret) + 1j * torch.cos(azim) * torch.sin(ret)
#     diag2 = torch.conj(diag1)
#     # Construct Jones Matrix
#     JM = torch.zeros([Delta_n.shape[0], 2, 2], dtype=torch.complex64, device=Delta_n.device)
#     JM[:,0,0] = diag1
#     JM[:,0,1] = offdiag
#     JM[:,1,0] = offdiag
#     JM[:,1,1] = diag2
    return JM

In [ ]:
backend = BackEnds.NUMPY

In [ ]:
LCP = JonesMatrixGenerators.left_circular_polarizer()

In [ ]:
m1 = JonesMatrixGenerators.linear_retarder(0.1, .2)
m2 = JonesMatrixGenerators.linear_retarder(0.1, .3)

In [ ]:
retardance(m1)

In [ ]:
ret = retardance(m1 @ m2)

In [ ]:
azim = azimuth(m1 @ m2)

In [ ]:
print(f"ret: {ret}, azim: {azim}")

In [ ]:
JM_list = [m1, m2, m1@m2, m2@m1]

In [ ]:
for JM in JM_list:
    ret = retardance(JM)
    azim = azimuth(JM)
    print(f"ret: {ret}, azim: {azim}")

In [25]:
# v1 = voxRayJM(Delta_n, opticAxis, rayDir, ell, wavelength)
opticAxis = np.array([0,1,1]) / np.linalg.norm(np.array([0,1,1]))
v1 = voxRayJM(0.01, opticAxis, [np.array([1,0,0]), np.array([0,1,0]), np.array([0,0,1])], 1, 0.5)
v2 = voxRayJM(0.03, opticAxis, [np.array([1,0,0]), np.array([0,1,0]), np.array([0,0,1])], 1, 0.5)
opticAxis = np.array([0,2,3]) / np.linalg.norm(np.array([0,2,3]))
v3 = voxRayJM(0.01, opticAxis, [np.array([1,0,0]), np.array([0,1,0]), np.array([0,0,1])], 1, 0.5)
v4 = voxRayJM(0.02, opticAxis, [np.array([1,0,0]), np.array([0,1,0]), np.array([0,0,1])], 1, 0.5)
v5 = voxRayJM(0.03, opticAxis, [np.array([1,0,0]), np.array([0,1,0]), np.array([0,0,1])], 1, 0.5)

In [27]:
JM_list = [v1, v2, v3, v4, v5, v1@v2, v1@v3, v1@v4, v1@v5]
for JM in JM_list:
    ret = retardance(JM)
    azim = azimuth(JM)
    print(f"ret: {ret}, azim: {azim}")

ret: 0.1256637061435916, azim: 0.7853981633974483
ret: 0.3769911184307752, azim: 0.7853981633974483
ret: 0.12566370614359154, azim: 0.9827937232473292
ret: 0.25132741228718347, azim: 0.982793723247329
ret: 0.376991118430775, azim: 0.982793723247329
ret: 0.5026548245743668, azim: 0.7853981633974483
ret: 0.24644055757359123, azim: 0.8840959433223887
ret: 0.3704741350999817, azim: 0.9175553965382035
ret: 0.4953238622751753, azim: 0.9343237024278891


In [28]:
JM_list = [v2@v1, v3@v1, v4@v1, v5@v1]
for JM in JM_list:
    ret = retardance(JM)
    azim = azimuth(JM)
    print(f"ret: {ret}, azim: {azim}")

ret: 0.5026548245743668, azim: 0.7853981633974483
ret: 0.24644055757359112, azim: 0.8840959433223887
ret: 0.3704741350999817, azim: 0.9175553965382035
ret: 0.49532386227517533, azim: 0.9343237024278891


In [47]:
Ein = np.array([1,0])
JM_list = [v1, v2, v3, v4, v5, v1@v2, v1@v3, v1@v4, v1@v5]
for JM in JM_list:
    ret = retardance(JM)
    azim = azimuth(JM)
    Eout = JM @ Ein
    print(f"ret: {ret:.2f}, azim: {azim:.2f}; Eout: {Eout[0]:.3f}, {Eout[1]:.3f}")

ret: 0.13, azim: 0.79; Eout: 0.998+0.000j, 0.000+0.063j
ret: 0.38, azim: 0.79; Eout: 0.982+0.000j, 0.000+0.187j
ret: 0.13, azim: 0.98; Eout: 0.998+0.024j, -0.000+0.058j
ret: 0.25, azim: 0.98; Eout: 0.992+0.048j, -0.000+0.116j
ret: 0.38, azim: 0.98; Eout: 0.982+0.072j, -0.000+0.173j
ret: 0.50, azim: 0.79; Eout: 0.969+0.000j, 0.000+0.249j
ret: 0.25, azim: 0.88; Eout: 0.992+0.024j, -0.002+0.121j
ret: 0.37, azim: 0.92; Eout: 0.983+0.048j, -0.003+0.178j
ret: 0.50, azim: 0.93; Eout: 0.969+0.072j, -0.005+0.234j


In [46]:
Ein = np.array([1,1j]) / np.sqrt(2)
JM_list = [v1, v2, v3, v4, v5, v1@v2, v1@v3, v1@v4, v1@v5]
for JM in JM_list:
    ret = retardance(JM)
    azim = azimuth(JM)
    Eout = JM @ Ein
    print(f"ret: {ret:.2f}, azim: {azim:.2f}; Eout: {Eout[0]:.3f}, {Eout[1]:.3f}, Enorm: {np.linalg.norm(Eout):.2f}")

ret: 0.13, azim: 0.79; Eout: 0.661+0.000j, 0.000+0.750j, Enorm: 1.00
ret: 0.38, azim: 0.79; Eout: 0.562+0.000j, 0.000+0.827j, Enorm: 1.00
ret: 0.13, azim: 0.98; Eout: 0.665+0.017j, 0.017+0.747j, Enorm: 1.00
ret: 0.25, azim: 0.98; Eout: 0.620+0.034j, 0.034+0.783j, Enorm: 1.00
ret: 0.38, azim: 0.98; Eout: 0.572+0.051j, 0.051+0.817j, Enorm: 1.00
ret: 0.50, azim: 0.79; Eout: 0.509+0.000j, 0.000+0.861j, Enorm: 1.00
ret: 0.25, azim: 0.88; Eout: 0.617+0.018j, 0.016+0.787j, Enorm: 1.00
ret: 0.37, azim: 0.92; Eout: 0.569+0.036j, 0.032+0.821j, Enorm: 1.00
ret: 0.50, azim: 0.93; Eout: 0.520+0.054j, 0.048+0.851j, Enorm: 1.00


In [35]:
JM_list = [v1@v5@v1@v4, v5@v1@v1@v4, v1@v1@v5@v4]
for JM in JM_list:
    ret = retardance(JM)
    azim = azimuth(JM)
    print(f"ret: {ret:.2f}, azim: {azim:.2f}")

ret: 0.87, azim: 0.93
ret: 0.87, azim: 0.93
ret: 0.87, azim: 0.93


In [11]:
import numpy as np
from VolumeRaytraceLFM.birefringence_implementations import  (
    JonesMatrixGenerators
)
def retardance(JM):
    '''Phase delay introduced between the fast and slow axis in a Jones Matrix'''
    e1, e2 = np.linalg.eigvals(JM)
    phase_diff = np.angle(e1) - np.angle(e2)
    retardance = np.abs(phase_diff)
    return retardance

def azimuth(JM):
    '''Rotation angle of the fast axis (neg phase)'''
    diag_sum = JM[0, 0] + JM[1, 1]
    diag_diff = JM[1, 1] - JM[0, 0]
    off_diag_sum = JM[0, 1] + JM[1, 0]
    a = np.imag(diag_diff / diag_sum)
    b = np.imag(off_diag_sum / diag_sum)
    azimuth = np.arctan2(-b, -a) / 2 + np.pi / 2
    return azimuth

In [2]:
def voxRayJM(Delta_n, opticAxis, rayDir, ell, wavelength):
    '''Compute Jones matrix associated with a particular ray and voxel combination'''
    # Azimuth is the angle of the slow axis of retardance.
    azim = np.arctan2(np.dot(opticAxis, rayDir[1]), np.dot(opticAxis, rayDir[2]))
    if Delta_n == 0:
        azim = 0
    elif Delta_n < 0:
        azim = azim + np.pi / 2
    ret = abs(Delta_n) * (1 - np.dot(opticAxis, rayDir[0]) ** 2) * 2 * np.pi * ell / wavelength
    JM = JonesMatrixGenerators.linear_retarder(ret, azim)
    return JM

In [18]:
opticAxis = np.array([1,1,0]) /  np.linalg.norm(np.array([1,1,0]))
JM = voxRayJM(0.01, opticAxis, [np.array([1,0,0]), np.array([0,1,0]), np.array([0,0,1])], 1, 0.5)
ret = retardance(JM)
azim = azimuth(JM)
print(f"ret: {ret}, azim: {azim}")

ret: 0.5026548245743668, azim: 0.7853981633974483


In [ ]:
opticAxis = np.array([1,1,0]) /  np.linalg.norm(np.array([1,1,0]))
JM1 = voxRayJM(0.01, opticAxis, [np.array([1,0,0]), np.array([0,1,0]), np.array([0,0,1])], 1, 0.5)
opticAxis = np.array([1,1,0]) /  np.linalg.norm(np.array([1,1,0]))
JM2 = voxRayJM(0.01, opticAxis, [np.array([1,0,0]), np.array([0,1,0]), np.array([0,0,1])], 1, 0.5)
ret = retardance(JM)
azim = azimuth(JM)
print(f"ret: {ret}, azim: {azim}")